In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, avg, desc, count

# 1. CONFIGURACIÓN DE LA SESIÓN DE SPARK
# Incluimos el conector de MongoDB necesario para la comunicación
spark = SparkSession.builder \
    .appName("AnalisisAutosNewkar") \
    .config("spark.mongodb.input.uri", "mongodb://mongodb:27017/proyecto_bigdata.AutoTec") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

try:
    # 2. CARGA DE DATOS DESDE MONGODB
    print("📥 Cargando datos desde MongoDB...")
    df = spark.read.format("com.mongodb.spark.sql.DefaultSource").load()

    if df.count() == 0:
        print("❌ La colección está vacía. Ejecuta el scraper primero.")
    else:
        print(f"✅ Se cargaron {df.count()} registros correctamente.")

        # 3. TRANSFORMACIÓN: Extraer la Marca y limpiar
        # Tomamos la primera palabra del 'identificador' como la Marca del auto
        df_con_marca = df.withColumn("marca", split(col("identificador"), " ")[0])

        # 4. LIMPIEZA: Filtrar precios coherentes
        # Eliminamos registros con valor 0 o muy bajos que puedan ser errores de carga
        df_limpio = df_con_marca.filter(col("valor") > 100000)

        # 5. ANÁLISIS 1: Ver los 10 autos más caros
        print("\n--- TOP 10 AUTOS MÁS CAROS EN NEWKAR ---")
        df_limpio.select("marca", "identificador", "valor") \
            .orderBy(col("valor").desc()) \
            .show(10, truncate=False)

        # 6. ANÁLISIS 2: Reporte por Marca (Cantidad y Precio Promedio)
        print("\n--- REPORTE POR MARCA ---")
        reporte = df_limpio.groupBy("marca").agg(
            count("identificador").alias("cantidad"),
            avg("valor").alias("precio_promedio")
        ).orderBy(col("precio_promedio").desc())

        reporte.show()

        # 7. ESTADÍSTICAS GENERALES
        precio_max = df_limpio.agg({"valor": "max"}).collect()[0][0]
        print(f"\n💰 El auto más caro del inventario cuesta: ${precio_max:,.0f}")

except Exception as e:
    print(f"❌ Error durante el procesamiento Spark: {e}")

# (Opcional) Detener la sesión al finalizar
# spark.stop()

📥 Cargando datos desde MongoDB...
✅ Se cargaron 28 registros correctamente.

--- TOP 10 AUTOS MÁS CAROS EN NEWKAR ---
+----------+---------------------------+-------+
|marca     |identificador              |valor  |
+----------+---------------------------+-------+
|Mazda     |Mazda CX-30 2026           |2.149E7|
|MAZDA     |MAZDA CX-30 2024           |2.099E7|
|Haval     |Haval H6 GT 2023           |2.049E7|
|Toyota    |Toyota Hilux 2021          |1.919E7|
|RAM       |RAM 1000 2022              |1.649E7|
|Toyota    |Toyota Corolla XEI 2.0 2023|1.599E7|
|Chevrolet |Chevrolet Montana 1.2 2024 |1.549E7|
|Toyota    |Toyota Yaris 1.5 2024      |1.499E7|
|Volkswagen|Volkswagen Tiguan 1.4 2019 |1.459E7|
|Volvo     |Volvo S60 T4 R-DESIGN 2017 |1.449E7|
+----------+---------------------------+-------+
only showing top 10 rows


--- REPORTE POR MARCA ---
+----------+--------+--------------------+
|     marca|cantidad|     precio_promedio|
+----------+--------+--------------------+
|     MAZDA|  